# Visualization map

Notebook nay chi ve ban do tu 2 file dau vao: obstacle grid va radiation grid. Khong load hoac ve path.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from pathlib import Path

# -------------------------------------------------
# CAU HINH DAU VAO (chinh tai day)
# -------------------------------------------------
# OBSTACLE_FILE  = "data/maps/scenario_medium/obstacle_grid.txt"
# RADIATION_FILE = "data/maps/scenario_medium/radiation_grid.txt"

# Vi du cac map khac:
# OBSTACLE_FILE  = "data/maps/mixed200/mixed2002.txt"
# #RADIATION_FILE = "data/maps/mixed200/radiation_grid.txt"
# RADIATION_FILE = "data/maps/generated_radiation_grid.txt"

# OBSTACLE_FILE  = "data/maps/triangle300/triangle300.txt"
# RADIATION_FILE = "data/maps/triangle300/radiation_grid_triangle300.txt"

OBSTACLE_FILE  = "data/maps/square400/square400.txt"
RADIATION_FILE = "data/maps/square400/radiation_grid.txt"

# OBSTACLE_FILE  = "data/maps/factory400/factory400_30_40.txt"
# RADIATION_FILE = "data/maps/factory400/radiation_grid_30.txt"

# OBSTACLE_FILE  = "data/maps/mixed500/mixed500.txt"
# RADIATION_FILE = "data/maps/mixed500/radiation_grid.txt"

# OBSTACLE_FILE = "data/scenario7/scenario7_grid.txt"
# RADIATION_FILE = "data/scenario7/radiation_grid.txt"


SAVE_PATH       = None   # None = chi hien thi; dat duong dan .png de luu
SHOW_GRID_LINES = True   # ke o luoi
RI_MAX          = 8.0    # nguong phan loai high risk
LOW_THRESH      = 0.5    # nguong phan loai medium risk

In [2]:
# -------------------------------------------------
# DOC FILE
# -------------------------------------------------
def load_txt(path, dtype=float):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append([dtype(x) for x in line.split()])
    return np.array(rows, dtype=dtype)

obstacle  = load_txt(OBSTACLE_FILE,  dtype=int)
radiation = load_txt(RADIATION_FILE, dtype=float)

assert obstacle.shape == radiation.shape, "Grid size mismatch!"
nrows, ncols = obstacle.shape

print(f"Map size: {nrows} x {ncols}")
print(f"Targets (value 2): {(obstacle == 2).sum()}")
print(f"Obstacles (value 1): {(obstacle == 1).sum()}")
print(f"Radiation min: {radiation.min():.2f}")
print(f"Radiation max: {radiation.max():.2f}")
print(f"Cells >= RI_MAX: {(radiation >= RI_MAX).sum()}")

Map size: 400 x 400
Targets (value 2): 24
Obstacles (value 1): 94550
Radiation min: 0.04
Radiation max: 20.71
Cells >= RI_MAX: 743


In [3]:
# -------------------------------------------------
# THONG SO CUA OBSTACLE_FILE
# -------------------------------------------------
n_rows, n_cols = obstacle.shape
total_cells    = n_rows * n_cols

n_obstacles    = int((obstacle == 1).sum())   # so o vat can (gia tri 1)
n_waypoints    = int((obstacle == 2).sum())   # so waypoint (gia tri 2)
obstacle_density = n_obstacles / total_cells  # mat do vat can

print(f"File          : {OBSTACLE_FILE}")
print(f"Kich thuoc map: {n_rows} x {n_cols}  ({total_cells} o)")
print(f"So vat can    : {n_obstacles}")
print(f"Mat do vat can: {obstacle_density:.4f}  ({obstacle_density * 100:.2f}%)")
print(f"So waypoint   : {n_waypoints}")

File          : data/maps/square400/square400.txt
Kich thuoc map: 400 x 400  (160000 o)
So vat can    : 94550
Mat do vat can: 0.5909  (59.09%)
So waypoint   : 24


In [4]:
# -------------------------------------------------
# VE BAN DO (khong ve path)
# -------------------------------------------------
# Phan loai tung o: 0=obstacle, 1=low, 2=medium, 3=high risk
base = np.ones_like(obstacle, dtype=int)          # default: low risk
base[radiation >= LOW_THRESH] = 2                # medium risk
base[radiation >= RI_MAX]     = 3                # high risk
base[obstacle == 1]           = 0                # obstacle

cmap = ListedColormap([
    "#222222",  # 0: obstacle
    "#87CEEB",  # 1: low risk - sky blue
    "#FFFF66",  # 2: med risk - yellow
    "#FFB6B6",  # 3: high risk - pink
])

fig_w = max(10, ncols * 0.28)
fig_h = max(8,  nrows * 0.28)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

# Nen ban do: dat extent theo bien o de toa do tam o la (col, row)
ax.imshow(base, cmap=cmap, vmin=0, vmax=3,
          origin="upper", interpolation="none",
          extent=(-0.5, ncols - 0.5, nrows - 0.5, -0.5),
          zorder=0)

# Luoi o: ve truc tiep tren bien tung cell, on dinh hon ax.grid/minor ticks
if SHOW_GRID_LINES:
    x_edges = np.arange(-0.5, ncols + 0.5, 1)
    y_edges = np.arange(-0.5, nrows + 0.5, 1)
    ax.vlines(x_edges, ymin=-0.5, ymax=nrows - 0.5,
              colors="black", linewidth=0.45, alpha=0.75,
              zorder=4, clip_on=False)
    ax.hlines(y_edges, xmin=-0.5, xmax=ncols - 0.5,
              colors="black", linewidth=0.45, alpha=0.75,
              zorder=4, clip_on=False)

ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(which="both", bottom=False, left=False,
               labelbottom=False, labelleft=False)

# Targets (ngoi sao xanh) neu obstacle map co gia tri 2
target_pos = np.argwhere(obstacle == 2)
for idx, (r, c) in enumerate(target_pos, start=1):
    ax.scatter(c, r, marker="*", s=600, c="blue",
               edgecolors="blue", linewidths=0.8, zorder=8)
    ax.text(c + 0.3, r + 0.3, str(idx),
            color="blue", fontsize=9, fontweight="bold",
            ha="left", va="center", zorder=9)

ax.set_xlim(-0.5, ncols - 0.5)
ax.set_ylim(nrows - 0.5, -0.5)
ax.set_aspect("equal")

# Legend giong visualization.ipynb, nhung bo path/start/end
# legend_items = [
#     mpatches.Patch(color="#222222", label="Obstacle"),
#     mpatches.Patch(color="#87CEEB", label="Low risk"),
#     mpatches.Patch(color="#FFFF66", label="Medium risk"),
#     mpatches.Patch(color="#FFB6B6", label="High risk"),
#     plt.Line2D([0], [0], marker="*", color="blue", linestyle="None",
#                markersize=10, label="Target"),
# ]
# ax.legend(handles=legend_items, loc="upper right",
#           fontsize=8, framealpha=0.85)

fig.tight_layout()

if SAVE_PATH:
    Path(SAVE_PATH).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(SAVE_PATH, dpi=150, bbox_inches="tight")
    print(f"Saved: {SAVE_PATH}")

plt.show()

In [5]:
# # -------------------------------------------------
# # VE OBSTACLE GRID DANG MA TRAN SO
# # -------------------------------------------------
# fig_w = max(10, ncols * 0.34)
# fig_h = max(8,  nrows * 0.34)
# fig, ax = plt.subplots(figsize=(fig_w, fig_h))

# obstacle_cmap = ListedColormap([
#     "#FFFFFF",  # 0: free
#     "#111111",  # 1: obstacle
#     "#D6ECFF",  # 2: target/checkpoint
# ])

# ax.imshow(obstacle, cmap=obstacle_cmap, vmin=0, vmax=2,
#           origin="upper", interpolation="none",
#           extent=(-0.5, ncols - 0.5, nrows - 0.5, -0.5),
#           zorder=0)

# x_edges = np.arange(-0.5, ncols + 0.5, 1)
# y_edges = np.arange(-0.5, nrows + 0.5, 1)
# ax.vlines(x_edges, ymin=-0.5, ymax=nrows - 0.5,
#           colors="#555555", linewidth=0.45, alpha=0.8,
#           zorder=3, clip_on=False)
# ax.hlines(y_edges, xmin=-0.5, xmax=ncols - 0.5,
#           colors="#555555", linewidth=0.45, alpha=0.8,
#           zorder=3, clip_on=False)

# font_size = max(6, min(11, 180 / max(nrows, ncols)))
# for r in range(nrows):
#     for c in range(ncols):
#         val = int(obstacle[r, c])
#         color = "white" if val == 1 else "black"
#         ax.text(c, r, str(val), ha="center", va="center",
#                 fontsize=font_size, color=color, zorder=4)

# ax.set_xlim(-0.5, ncols - 0.5)
# ax.set_ylim(nrows - 0.5, -0.5)
# ax.set_aspect("equal")
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_title("Obstacle grid", fontsize=14)
# fig.tight_layout()
# plt.show()

In [6]:
# # -------------------------------------------------
# # VE RADIATION GRID DANG MA TRAN SO
# # -------------------------------------------------
# # Phan loai tung o: 0=low, 1=medium, 2=high risk
# rad_class = np.zeros_like(radiation, dtype=int)
# rad_class[radiation >= LOW_THRESH] = 1
# rad_class[radiation >= RI_MAX] = 2

# radiation_cmap = ListedColormap([
#     "#87CEEB",  # low risk
#     "#FFFF99",  # medium risk
#     "#FF7F8A",  # high risk
# ])

# fig_w = max(10, ncols * 0.34)
# fig_h = max(8,  nrows * 0.34)
# fig, ax = plt.subplots(figsize=(fig_w, fig_h))

# ax.imshow(rad_class, cmap=radiation_cmap, vmin=0, vmax=2,
#           origin="upper", interpolation="none",
#           extent=(-0.5, ncols - 0.5, nrows - 0.5, -0.5),
#           zorder=0)

# x_edges = np.arange(-0.5, ncols + 0.5, 1)
# y_edges = np.arange(-0.5, nrows + 0.5, 1)
# ax.vlines(x_edges, ymin=-0.5, ymax=nrows - 0.5,
#           colors="#555555", linewidth=0.45, alpha=0.55,
#           zorder=3, clip_on=False)
# ax.hlines(y_edges, xmin=-0.5, xmax=ncols - 0.5,
#           colors="#555555", linewidth=0.45, alpha=0.55,
#           zorder=3, clip_on=False)

# font_size = max(6, min(10, 165 / max(nrows, ncols)))
# for r in range(nrows):
#     for c in range(ncols):
#         ax.text(c, r, f"{radiation[r, c]:.2f}", ha="center", va="center",
#                 fontsize=font_size, color="#1f3a4a", zorder=4)

# ax.set_xlim(-0.5, ncols - 0.5)
# ax.set_ylim(nrows - 0.5, -0.5)
# ax.set_aspect("equal")
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_title("Radiation grid", fontsize=14)
# fig.tight_layout()
# plt.show()